# RR Pressure Reliability & Auction Intelligence — Colab Notebook

This notebook processes the supplied `ball_by_ball.csv` and generates a role-aware ranking and a performance-only auction shortlist for the 20 nominated players.

## What the pipeline does
1. Reconstructs match winners from innings 1 and 2 totals. Six Super Over matches in this file are resolved from innings 3 and 4. Incomplete matches and unresolved ties are excluded from WP training.
2. Estimates the batting side's win probability before and after each delivery. The bowling side's win probability is `1 - batting-side WP`.
3. Uses GroupKFold out-of-fold predictions grouped by `match_id`, so deliveries from the same match do not appear in both training and validation for that prediction.
4. Defines pressure as a team's estimated pre-delivery win probability below the configurable threshold (default 0.40).
5. Computes a Pressure Leverage Index (PLI) for a pressure-side player only when that underdog team ultimately wins. PLI rewards positive win-probability added, weighted by how severe the underdog state was.
6. Produces batter, bowler and all-rounder reliability scores with sample-size shrinkage, then creates the 20-player ranking and performance-only auction recommendations.

## Pressure Leverage Index
For each successful underdog pressure delivery:

`severity = (threshold - team_WP_before) / threshold`

`positive_WPA = max(team_WP_after - team_WP_before, 0)`

`PLI per 100 pressure deliveries = 100 × sum(severity × positive_WPA) / number of successful underdog pressure deliveries`

The code stores each event's `100 × severity × positive_WPA` and averages it across comeback-pressure deliveries. Batting PLI is credited only when the striker faced the delivery (a wide does not count as a batter contribution). Bowling PLI can include wides/no-balls because the bowler delivered them.

## Important interpretation limits
- The CSV does not contain an explicit match-winner field or a target column; winners and chase targets are reconstructed from innings scores.
- Win probability is a model estimate, not bookmaker odds or official live win probability. The first-innings WP model is inherently harder to estimate than a chase model; use the validation metrics printed by the notebook when interpreting its 0.40 threshold.
- PLI is a team-state leverage proxy attributed to the active striker/bowler on a delivery. It does not prove one player alone caused a comeback.
- Auction recommendations are performance-only. IPL 2027 availability, eligibility, role need, price, squad balance and retention constraints are not in this CSV and must be researched separately.

## How to run
Run the cells from top to bottom. When prompted, upload `ball_by_ball.csv` (about 40 MB). At the end, the notebook downloads a ZIP containing the ranking, auction recommendation, WP validation, inferred results and pressure-event CSV files.


## Cell 1: SETTINGS


In [1]:
import os, re, zipfile, warnings
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

warnings.filterwarnings("ignore", category=UserWarning)
DATA_PATH = os.environ.get("DATA_PATH", "ball_by_ball.csv")

# Configurable pressure threshold suggested in the hackathon discussion.
PRESSURE_THRESHOLD = 0.40
N_SPLITS = 5
RANDOM_STATE = 42
MAX_INNINGS_BALLS = 120   # 20-over T20 baseline; clipped for shortened/extra cases.

CANDIDATES = [
    "Rashid Khan", "Abdul Samad", "Haider Ali", "Basil Hameed",
    "Muhammad Farooq", "Bashir Ahmad", "Mahendra Singh Dhoni",
    "Piyush Chawla", "Ishant Sharma", "Nathan Bracken", "Virat Kohli",
    "Faf du Plessis", "Wriddhiman Saha", "Rohit Sharma", "Shikhar Dhawan",
    "Dinesh Karthik", "Pradeep Sangwan", "Ravindra Jadeja", "Amit Mishra",
    "Dwayne Bravo"
]

print(f"Input file: {DATA_PATH}")
print(f"Pressure threshold: team win probability < {PRESSURE_THRESHOLD:.2f}")




Input file: ball_by_ball.csv
Pressure threshold: team win probability < 0.40


## Cell 2: LOAD & VALIDATE


In [2]:
raw = pd.read_csv(DATA_PATH, low_memory=False)

REQUIRED_COLUMNS = [
    "match_id", "innings", "innings_ball_num", "batting_team_id",
    "bowling_team_id", "striker_id", "striker_name", "non_striker_id",
    "non_striker_name", "bowler_id", "bowler_name", "runs", "is_four",
    "is_six", "is_wicket", "dismissed_player_id", "dismissal_type",
    "extras_type", "extras1_runs", "extras_2_runs"
]
missing_cols = [c for c in REQUIRED_COLUMNS if c not in raw.columns]
if missing_cols:
    raise ValueError(f"The uploaded CSV is missing required columns: {missing_cols}")

# Normalize data types and create delivery-level total runs.
df = raw.copy()
for c in ["match_id", "innings", "innings_ball_num", "batting_team_id",
          "bowling_team_id", "striker_id", "non_striker_id", "bowler_id",
          "runs", "extras1_runs", "extras_2_runs"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

for c in ["is_four", "is_six", "is_wicket"]:
    df[c] = df[c].astype(str).str.strip().str.lower().map(
        {"true": True, "false": False, "1": True, "0": False}
    ).fillna(False)

for c in ["extras_type", "dismissal_type"]:
    df[c] = df[c].fillna("").astype(str).str.strip()

df["total_extras"] = df["extras1_runs"].fillna(0) + df["extras_2_runs"].fillna(0)
df["total_runs"] = df["runs"].fillna(0) + df["total_extras"]
df["extras_lower"] = df["extras_type"].str.lower()

# In this source, "wide" and "Overstep" (no-ball) are non-legal deliveries.
# Other extras (byes/leg-byes) are treated as legal deliveries.
df["is_wide"] = df["extras_lower"].eq("wide")
df["is_no_ball"] = df["extras_lower"].isin(["overstep", "no ball", "noball"])
df["legal_ball"] = ~(df["is_wide"] | df["is_no_ball"])

# Sort on the dataset's unique delivery sequence within each match innings.
df = df.sort_values(["match_id", "innings", "innings_ball_num"]).reset_index(drop=True)
print("Loaded shape:", df.shape)
print("Unique matches:", df["match_id"].nunique())
print("Date range (if timestamp exists):")
if "delivery_ist_time" in df.columns:
    dt = pd.to_datetime(df["delivery_ist_time"], errors="coerce", utc=True)
    print(" ", dt.min(), "to", dt.max())




Loaded shape: (200999, 39)
Unique matches: 867
Date range (if timestamp exists):
  2022-01-23 00:00:00+00:00 to 2026-11-01 16:50:46+00:00


## Cell 3: RECONSTRUCT MATCH WINNERS


In [3]:
# There is no explicit winner/result column in the CSV. We reconstruct the
# winner from innings 1 and 2 totals. Tied matches with innings 3 and 4 are
# resolved using Super Over scores; unresolved ties and incomplete matches
# are excluded from win-probability training.
innings_totals = (
    df.groupby(["match_id", "innings"], as_index=False)
      .agg(score=("total_runs", "sum"),
           wickets=("is_wicket", "sum"),
           batting_team_id=("batting_team_id", "first"),
           bowling_team_id=("bowling_team_id", "first"),
           delivery_count=("innings_ball_num", "size"))
)

innings_lookup = {
    (int(r.match_id), int(r.innings)): r
    for r in innings_totals.itertuples(index=False)
}
match_result_rows = []
for mid in sorted(df["match_id"].dropna().astype(int).unique()):
    i1 = innings_lookup.get((mid, 1))
    i2 = innings_lookup.get((mid, 2))
    if i1 is None or i2 is None:
        continue

    s1, s2 = float(i1.score), float(i2.score)
    winner_id, method = np.nan, ""
    if s1 > s2:
        winner_id, method = int(i1.batting_team_id), "regulation"
    elif s2 > s1:
        winner_id, method = int(i2.batting_team_id), "regulation"
    else:
        i3 = innings_lookup.get((mid, 3))
        i4 = innings_lookup.get((mid, 4))
        if i3 is not None and i4 is not None:
            if float(i3.score) > float(i4.score):
                winner_id, method = int(i3.batting_team_id), "super_over"
            elif float(i4.score) > float(i3.score):
                winner_id, method = int(i4.batting_team_id), "super_over"
            else:
                method = "unresolved_super_over_tie"
        else:
            method = "unresolved_regulation_tie"

    if pd.notna(winner_id):
        match_result_rows.append({
            "match_id": mid,
            "winner_team_id": int(winner_id),
            "innings1_score": s1,
            "innings2_score": s2,
            "result_method": method
        })

match_results = pd.DataFrame(match_result_rows)
if match_results.empty:
    raise RuntimeError("No completed match results could be reconstructed.")

print("Completed matches with inferred winner:", len(match_results))
print("Result methods:\n", match_results["result_method"].value_counts().to_string())
print("Matches excluded (incomplete/unresolved):",
      df["match_id"].nunique() - len(match_results))




Completed matches with inferred winner: 842
Result methods:
 result_method
regulation    836
super_over      6
Matches excluded (incomplete/unresolved): 25


## Cell 4: BUILD PRE/POST MATCH STATES


In [4]:
# Only regulation innings 1 and 2 are used to estimate standard T20 WP.
# Innings 3/4 are used only to resolve tied match results.
wp = df[df["innings"].isin([1, 2])].merge(
    match_results, on="match_id", how="inner", validate="many_to_one"
).copy()
wp = wp.sort_values(["match_id", "innings", "innings_ball_num"]).reset_index(drop=True)

wp["batting_team_won"] = (wp["batting_team_id"] == wp["winner_team_id"]).astype(int)
wp["bowling_team_won"] = (wp["bowling_team_id"] == wp["winner_team_id"]).astype(int)

# Target is the first innings total + 1, available to the chasing team.
first_innings_scores = (
    innings_totals[innings_totals["innings"] == 1]
    .set_index("match_id")["score"]
    .to_dict()
)
wp["target"] = np.where(
    wp["innings"].eq(2),
    wp["match_id"].map(first_innings_scores).astype(float) + 1.0,
    np.nan
)

# Pre-delivery state: cumulative totals shifted by one delivery.
grp = wp.groupby(["match_id", "innings"], sort=False)
wp["score_after"] = grp["total_runs"].cumsum()
wp["wickets_after"] = grp["is_wicket"].cumsum().astype(float)
wp["legal_balls_after"] = grp["legal_ball"].cumsum().astype(float)
wp["score_before"] = wp["score_after"] - wp["total_runs"]
wp["wickets_before"] = wp["wickets_after"] - wp["is_wicket"].astype(float)
wp["legal_balls_before"] = wp["legal_balls_after"] - wp["legal_ball"].astype(float)

# Recent form features use the last six recorded deliveries (not necessarily
# six legal balls, because wides/no-balls are rows in the source).
grp = wp.groupby(["match_id", "innings"], sort=False)
wp["recent_runs_pre"] = grp["total_runs"].transform(
    lambda s: s.shift(1).rolling(6, min_periods=1).sum()
).fillna(0)
wp["recent_wickets_pre"] = grp["is_wicket"].transform(
    lambda s: s.shift(1).rolling(6, min_periods=1).sum()
).fillna(0)
wp["recent_runs_post"] = grp["total_runs"].transform(
    lambda s: s.rolling(6, min_periods=1).sum()
).fillna(0)
wp["recent_wickets_post"] = grp["is_wicket"].transform(
    lambda s: s.rolling(6, min_periods=1).sum()
).fillna(0)

# Mark final delivery in each innings.
wp["is_last_delivery_in_innings"] = (
    wp.groupby(["match_id", "innings"]).cumcount(ascending=False).eq(0)
)

FEATURES_INN1 = [
    "score", "wickets_lost", "legal_balls", "balls_remaining",
    "wickets_in_hand", "current_run_rate", "smoothed_run_rate",
    "projected_total", "recent_runs_6", "recent_wickets_6",
    "is_powerplay", "is_death"
]
FEATURES_INN2 = [
    "score", "wickets_lost", "legal_balls", "balls_remaining",
    "wickets_in_hand", "current_run_rate", "smoothed_run_rate",
    "target", "runs_required", "required_run_rate", "chase_progress",
    "recent_runs_6", "recent_wickets_6", "is_powerplay", "is_death"
]

def make_state_features(frame, state="pre"):
    suffix = "before" if state == "pre" else "after"
    score = frame[f"score_{suffix}"]
    wickets = frame[f"wickets_{suffix}"]
    legal_balls = frame[f"legal_balls_{suffix}"]
    balls_remaining = (MAX_INNINGS_BALLS - legal_balls).clip(0, MAX_INNINGS_BALLS)
    overs_bowled = legal_balls / 6.0
    current_rr = np.where(overs_bowled > 0, score / np.maximum(overs_bowled, 1e-6), 0.0)
    # A light 20-run / 18-ball prior stabilises projected scores early in an innings.
    smoothed_rr = 6.0 * (score + 20.0) / (legal_balls + 18.0)
    recent_runs = frame["recent_runs_pre" if state == "pre" else "recent_runs_post"]
    recent_wkts = frame["recent_wickets_pre" if state == "pre" else "recent_wickets_post"]

    x = pd.DataFrame(index=frame.index)
    x["score"] = score.astype(float)
    x["wickets_lost"] = wickets.astype(float).clip(0, 10)
    x["legal_balls"] = legal_balls.astype(float).clip(0, MAX_INNINGS_BALLS)
    x["balls_remaining"] = balls_remaining.astype(float)
    x["wickets_in_hand"] = (10.0 - wickets).clip(0, 10)
    x["current_run_rate"] = np.clip(current_rr, 0, 36)
    x["smoothed_run_rate"] = np.clip(smoothed_rr, 0, 36)
    x["recent_runs_6"] = recent_runs.astype(float).clip(0, 60)
    x["recent_wickets_6"] = recent_wkts.astype(float).clip(0, 6)
    x["is_powerplay"] = (legal_balls < 36).astype(int)
    x["is_death"] = (legal_balls >= 90).astype(int)

    if state == "pre":
        inns = frame["innings"]
    else:
        inns = frame["innings"]

    # First-innings and chase states use separate models.
    x["projected_total"] = (score + smoothed_rr * balls_remaining / 6.0).clip(0, 400)
    if "target" in frame.columns:
        target = frame["target"].fillna(0).astype(float)
    else:
        target = pd.Series(0.0, index=frame.index)
    runs_required = (target - score).clip(lower=0)
    x["target"] = target
    x["runs_required"] = runs_required
    x["required_run_rate"] = np.where(
        balls_remaining > 0, runs_required * 6.0 / np.maximum(balls_remaining, 1),
        np.where(runs_required > 0, 36.0, 0.0)
    )
    x["required_run_rate"] = x["required_run_rate"].clip(0, 36)
    x["chase_progress"] = np.where(target > 0, score / target, 0.0).clip(0, 2)

    # Return the correct feature columns for this innings.
    return x

X_pre_all = make_state_features(wp, "pre")
X_post_all = make_state_features(wp, "post")




## Cell 5: OUT-OF-FOLD WIN PROBABILITY


In [5]:
# Each match is held out in its entirety. The model never sees another ball
# from the same match while predicting that match's WP.
wp["wp_batting_pre"] = np.nan
wp["wp_batting_post"] = np.nan
validation_rows = []

model_params = dict(
    max_iter=180,
    learning_rate=0.07,
    max_leaf_nodes=20,
    min_samples_leaf=80,
    l2_regularization=1.0,
    random_state=RANDOM_STATE
)

for inn in [1, 2]:
    idx = wp.index[wp["innings"] == inn].to_numpy()
    X_inn = X_pre_all.loc[idx, FEATURES_INN1 if inn == 1 else FEATURES_INN2]
    X_post_inn = X_post_all.loc[idx, FEATURES_INN1 if inn == 1 else FEATURES_INN2]
    y_inn = wp.loc[idx, "batting_team_won"].astype(int).to_numpy()
    groups_inn = wp.loc[idx, "match_id"].to_numpy()

    gkf = GroupKFold(n_splits=N_SPLITS)
    for fold, (tr, va) in enumerate(gkf.split(X_inn, y_inn, groups=groups_inn), start=1):
        y_train = y_inn[tr]
        if len(np.unique(y_train)) < 2:
            raise RuntimeError(f"Fold {fold}, innings {inn}: training fold has only one class.")

        model = HistGradientBoostingClassifier(**model_params)
        model.fit(X_inn.iloc[tr], y_train)
        p_pre = model.predict_proba(X_inn.iloc[va])[:, 1]
        p_post = model.predict_proba(X_post_inn.iloc[va])[:, 1]
        wp.loc[idx[va], "wp_batting_pre"] = p_pre
        wp.loc[idx[va], "wp_batting_post"] = p_post

    valid_mask = wp["innings"].eq(inn) & wp["wp_batting_pre"].notna()
    y_true = wp.loc[valid_mask, "batting_team_won"].astype(int)
    p_hat = wp.loc[valid_mask, "wp_batting_pre"].clip(1e-6, 1 - 1e-6)
    validation_rows.append({
        "innings": inn,
        "rows": int(valid_mask.sum()),
        "matches": int(wp.loc[valid_mask, "match_id"].nunique()),
        "roc_auc": roc_auc_score(y_true, p_hat) if y_true.nunique() == 2 else np.nan,
        "brier_score": brier_score_loss(y_true, p_hat),
        "log_loss": log_loss(y_true, p_hat, labels=[0, 1])
    })

validation = pd.DataFrame(validation_rows)

# For the final regulation delivery of a second innings, the match outcome is
# known after the delivery. If regulation scores were tied and a Super Over was
# required, use 0.5 at the end of regulation rather than attributing the Super
# Over outcome to the last regulation delivery.
last_i2 = wp["is_last_delivery_in_innings"] & wp["innings"].eq(2)
reg_tie_super_over = wp["result_method"].eq("super_over")
wp.loc[last_i2 & ~reg_tie_super_over, "wp_batting_post"] = wp.loc[
    last_i2 & ~reg_tie_super_over, "batting_team_won"
].astype(float)
wp.loc[last_i2 & reg_tie_super_over, "wp_batting_post"] = 0.5

wp["wp_batting_pre"] = wp["wp_batting_pre"].clip(0.001, 0.999)
wp["wp_batting_post"] = wp["wp_batting_post"].clip(0.001, 0.999)
wp["wpa_batting"] = wp["wp_batting_post"] - wp["wp_batting_pre"]
wp["wp_bowling_pre"] = 1.0 - wp["wp_batting_pre"]
wp["wp_bowling_post"] = 1.0 - wp["wp_batting_post"]
wp["wpa_bowling"] = wp["wp_bowling_post"] - wp["wp_bowling_pre"]

print("\nOut-of-fold win-probability validation:")
print(validation.round(4).to_string(index=False))




  File "c:\Users\PAWAN\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\PAWAN\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PAWAN\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\PAWAN\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^



Out-of-fold win-probability validation:
 innings   rows  matches  roc_auc  brier_score  log_loss
       1 103824      842   0.6660       0.2296    0.6538
       2  94860      842   0.8616       0.1552    0.4780


## Cell 6: PRESSURE + PLI


In [7]:
# Batting-side pressure: batting team's pre-delivery WP < threshold.
# Bowling-side pressure: bowling team's WP < threshold, equivalently batting
# team's WP > 1-threshold.
wp["batting_pressure"] = wp["wp_batting_pre"] < PRESSURE_THRESHOLD
wp["bowling_pressure"] = wp["wp_bowling_pre"] < PRESSURE_THRESHOLD
wp["batting_comeback_pressure"] = wp["batting_pressure"] & wp["batting_team_won"].eq(1)
wp["bowling_comeback_pressure"] = wp["bowling_pressure"] & wp["bowling_team_won"].eq(1)

wp["bat_pressure_severity"] = (
    (PRESSURE_THRESHOLD - wp["wp_batting_pre"]) / PRESSURE_THRESHOLD
).clip(0, 1)
wp["bowl_pressure_severity"] = (
    (PRESSURE_THRESHOLD - wp["wp_bowling_pre"]) / PRESSURE_THRESHOLD
).clip(0, 1)

# PLI event credit is positive win-probability added while the player's team
# was an underdog and ultimately won. It is a team-state leverage proxy, not
# proof that one individual alone caused the win.
wp["bat_positive_wpa"] = wp["wpa_batting"].clip(lower=0)
wp["bowl_positive_wpa"] = wp["wpa_bowling"].clip(lower=0)
wp["bat_pli_event"] = (
    wp["bat_pressure_severity"] * wp["bat_positive_wpa"] * 100.0
)
wp["bowl_pli_event"] = (
    wp["bowl_pressure_severity"] * wp["bowl_positive_wpa"] * 100.0
)

# Batting features. `runs` is the batter-run field; extras are not credited to
# the striker. A wide is not counted as a ball faced; a no-ball is counted as
# faced but not as a legal delivery.
wp["batter_ball_faced"] = ~wp["is_wide"]
wp["batter_dot"] = (
    wp["runs"].eq(0) & ~wp["is_wide"] & ~wp["is_no_ball"]
)
wp["batter_boundary"] = wp["is_four"] | wp["is_six"]

# Bowler runs conceded proxy: batter runs + extras charged to bowler. Byes,
# leg-byes and penalty extras are excluded from bowler runs conceded.
non_bowler_extras = wp["extras_lower"].isin(["byes", "legbyes", "penalty"])
wp["bowler_runs_conceded"] = wp["runs"].fillna(0) + np.where(
    non_bowler_extras, 0, wp["total_extras"]
)
wp["bowler_dot"] = wp["bowler_runs_conceded"].eq(0) & wp["legal_ball"]
wp["bowler_boundary_conceded"] = wp["is_four"] | wp["is_six"]
CREDITED_DISMISSALS = {"caught", "bowled", "lbw", "stump", "hit wicket"}
wp["bowler_credited_wicket"] = (
    wp["is_wicket"] & wp["dismissal_type"].str.lower().isin(CREDITED_DISMISSALS)
)




## Cell 7: PLAYER METRICS


In [8]:
# Build league-wide reference distributions first; candidate metrics are
# sample-size-shrunk toward these baselines before ranking.

# Batting under-pressure rows (all outcomes, not only comeback wins).
bat_p = wp[wp["batting_pressure"]].copy()
bat_p["player_id"] = bat_p["striker_id"].astype("Int64")
bat_p["player_name"] = bat_p["striker_name"].astype(str)
bat_p["batter_runs"] = bat_p["runs"].fillna(0)
bat_p["batter_faced"] = bat_p["batter_ball_faced"].astype(int)
bat_p["batter_dot_flag"] = bat_p["batter_dot"].astype(int)
bat_p["batter_boundary_flag"] = bat_p["batter_boundary"].astype(int)
bat_p["batting_comeback"] = bat_p["batting_comeback_pressure"]
bat_p["bat_pli_contribution"] = np.where(
    bat_p["batting_comeback"] & bat_p["batter_ball_faced"],
    bat_p["bat_pli_event"], 0.0
)

bat_basic = bat_p.groupby("player_id").agg(
    player_name=("player_name", "last"),
    bat_pressure_deliveries=("player_id", "size"),
    bat_pressure_balls_faced=("batter_faced", "sum"),
    bat_pressure_runs=("batter_runs", "sum"),
    bat_pressure_dots=("batter_dot_flag", "sum"),
    bat_pressure_boundaries=("batter_boundary_flag", "sum"),
    bat_pressure_matches=("match_id", "nunique"),
    bat_pli_points=("bat_pli_contribution", "sum")
).reset_index()

# Pressure innings consistency: share of pressure innings where the batter
# contributed at least 10 batter-runs during pressure deliveries.
bat_innings = (
    bat_p.groupby(["player_id", "match_id", "innings"], as_index=False)
         .agg(pressure_innings_runs=("batter_runs", "sum"))
)
bat_consistency = bat_innings.groupby("player_id").agg(
    bat_pressure_innings=("pressure_innings_runs", "size"),
    bat_10plus_pressure_innings=("pressure_innings_runs", lambda s: int((s >= 10).sum()))
).reset_index()
bat_consistency["bat_consistency_rate"] = (
    bat_consistency["bat_10plus_pressure_innings"] /
    bat_consistency["bat_pressure_innings"].replace(0, np.nan)
)

# Dismissals while the batting side was under pressure. Use dismissed_player_id
# so run-outs of the non-striker can also be assigned to the dismissed player.
dismissals = bat_p[bat_p["is_wicket"] & bat_p["dismissed_player_id"].notna()].copy()
dismissals["dismissed_player_id"] = pd.to_numeric(
    dismissals["dismissed_player_id"], errors="coerce"
)
dismissals = dismissals[dismissals["dismissed_player_id"].notna()]
bat_outs = dismissals.groupby("dismissed_player_id").size().rename("bat_pressure_dismissals").reset_index()
bat_outs = bat_outs.rename(columns={"dismissed_player_id": "player_id"})

# Successful comeback batting events for PLI.
bat_comeback = bat_p[bat_p["batting_comeback"]].copy()
bat_pli = bat_comeback.groupby("player_id").agg(
    bat_comeback_balls=("batter_faced", "sum"),
    bat_comeback_deliveries=("player_id", "size"),
    bat_comeback_matches=("match_id", "nunique"),
    bat_pli_sum=("bat_pli_event", "sum"),
    bat_comeback_wpa_sum=("bat_positive_wpa", "sum")
).reset_index()

# Bowling under-pressure rows (all outcomes).
bowl_p = wp[wp["bowling_pressure"]].copy()
bowl_p["player_id"] = bowl_p["bowler_id"].astype("Int64")
bowl_p["player_name"] = bowl_p["bowler_name"].astype(str)
bowl_p["bowl_comeback"] = bowl_p["bowling_comeback_pressure"]
bowl_p["bowl_pli_contribution"] = np.where(
    bowl_p["bowl_comeback"], bowl_p["bowl_pli_event"], 0.0
)

bowl_basic = bowl_p.groupby("player_id").agg(
    bowler_name=("player_name", "last"),
    bowl_pressure_deliveries=("player_id", "size"),
    bowl_pressure_legal_balls=("legal_ball", "sum"),
    bowl_pressure_runs_conceded=("bowler_runs_conceded", "sum"),
    bowl_pressure_dots=("bowler_dot", "sum"),
    bowl_pressure_boundaries=("bowler_boundary_conceded", "sum"),
    bowl_pressure_wickets=("bowler_credited_wicket", "sum"),
    bowl_pressure_matches=("match_id", "nunique"),
    bowl_pli_points=("bowl_pli_contribution", "sum")
).reset_index()

# Spell-level consistency: percentage of pressure spells with economy no worse
# than the league median pressure-spell economy, or at least one credited wicket.
bowl_spells = (
    bowl_p.groupby(["player_id", "match_id", "innings"], as_index=False)
          .agg(spell_runs=("bowler_runs_conceded", "sum"),
               spell_legal_balls=("legal_ball", "sum"),
               spell_wickets=("bowler_credited_wicket", "sum"))
)
bowl_spells["spell_economy"] = np.where(
    bowl_spells["spell_legal_balls"] > 0,
    bowl_spells["spell_runs"] * 6.0 / bowl_spells["spell_legal_balls"], np.nan
)
league_median_spell_econ = bowl_spells["spell_economy"].median()
bowl_spells["good_pressure_spell"] = (
    (bowl_spells["spell_economy"] <= league_median_spell_econ) |
    (bowl_spells["spell_wickets"] >= 1)
)
bowl_consistency = bowl_spells.groupby("player_id").agg(
    bowl_pressure_spells=("good_pressure_spell", "size"),
    bowl_good_pressure_spells=("good_pressure_spell", "sum")
).reset_index()
bowl_consistency["bowl_consistency_rate"] = (
    bowl_consistency["bowl_good_pressure_spells"] /
    bowl_consistency["bowl_pressure_spells"].replace(0, np.nan)
)

bowl_comeback = bowl_p[bowl_p["bowl_comeback"]].copy()
bowl_pli = bowl_comeback.groupby("player_id").agg(
    bowl_comeback_legal_balls=("legal_ball", "sum"),
    bowl_comeback_deliveries=("player_id", "size"),
    bowl_comeback_matches=("match_id", "nunique"),
    bowl_pli_sum=("bowl_pli_event", "sum"),
    bowl_comeback_wpa_sum=("bowl_positive_wpa", "sum")
).reset_index()

# Universe of IDs from striker, non-striker and bowler columns for name matching.
name_id_parts = []
for id_col, name_col in [("striker_id", "striker_name"),
                         ("non_striker_id", "non_striker_name"),
                         ("bowler_id", "bowler_name")]:
    tmp = df[[id_col, name_col]].dropna().copy()
    tmp.columns = ["player_id", "player_name"]
    tmp["player_id"] = pd.to_numeric(tmp["player_id"], errors="coerce")
    tmp = tmp.dropna(subset=["player_id"])
    tmp["player_id"] = tmp["player_id"].astype(int)
    name_id_parts.append(tmp)
name_id = pd.concat(name_id_parts, ignore_index=True).drop_duplicates()

def normalize_name(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

name_id["name_norm"] = name_id["player_name"].map(normalize_name)

# Build all-player volume counts for role classification.
bat_vol = df.groupby("striker_id").size().rename("total_batting_deliveries")
bowl_vol = df.groupby("bowler_id").size().rename("total_bowling_deliveries")
all_player_ids = set(name_id["player_id"].unique())
all_volume = pd.DataFrame({"player_id": sorted(all_player_ids)})
all_volume = all_volume.merge(bat_vol.rename_axis("player_id").reset_index(), on="player_id", how="left")
all_volume = all_volume.merge(bowl_vol.rename_axis("player_id").reset_index(), on="player_id", how="left")
all_volume[["total_batting_deliveries", "total_bowling_deliveries"]] = all_volume[
    ["total_batting_deliveries", "total_bowling_deliveries"]
].fillna(0)

# Merge batting and bowling pressure aggregates.
metrics = all_volume.merge(bat_basic, on="player_id", how="left")
metrics = metrics.merge(bat_consistency, on="player_id", how="left")
metrics = metrics.merge(bat_outs, on="player_id", how="left")
metrics = metrics.merge(bat_pli, on="player_id", how="left")
metrics = metrics.merge(bowl_basic, on="player_id", how="left")
metrics = metrics.merge(bowl_consistency, on="player_id", how="left")
metrics = metrics.merge(bowl_pli, on="player_id", how="left")

numeric_fill_cols = [c for c in metrics.columns if c not in ["player_id", "player_name", "bowler_name"]]
for c in numeric_fill_cols:
    metrics[c] = pd.to_numeric(metrics[c], errors="coerce").fillna(0)

# Resolve a display name from the most commonly recorded name for the ID.
name_by_id = (
    name_id.groupby(["player_id", "player_name"]).size().rename("n")
           .reset_index().sort_values(["player_id", "n"], ascending=[True, False])
           .drop_duplicates("player_id").set_index("player_id")["player_name"]
)
metrics["player_name"] = metrics["player_id"].map(name_by_id)

# League baselines and sample-size shrinkage.
league_bat = bat_p
league_bat_balls = max(int(league_bat["batter_faced"].sum()), 1)
prior_sr = float(league_bat["batter_runs"].sum() / league_bat_balls * 100)
prior_boundary_rate = float(league_bat["batter_boundary_flag"].sum() / league_bat_balls)
prior_dot_rate = float(league_bat["batter_dot_flag"].sum() / league_bat_balls)
prior_bat_avg = float(league_bat["batter_runs"].sum() / max(int(len(dismissals)), 1))

metrics["bat_sr_raw"] = np.where(
    metrics["bat_pressure_balls_faced"] > 0,
    metrics["bat_pressure_runs"] / metrics["bat_pressure_balls_faced"] * 100, np.nan
)
metrics["bat_sr_shrunk"] = (
    metrics["bat_pressure_runs"] + prior_sr / 100.0 * 30.0
) / (metrics["bat_pressure_balls_faced"] + 30.0) * 100.0
metrics["bat_avg_shrunk"] = (
    metrics["bat_pressure_runs"] + prior_bat_avg * 10.0
) / (metrics["bat_pressure_dismissals"] + 10.0)
metrics["bat_boundary_rate_shrunk"] = (
    metrics["bat_pressure_boundaries"] + prior_boundary_rate * 30.0
) / (metrics["bat_pressure_balls_faced"] + 30.0)
metrics["bat_dot_rate_shrunk"] = (
    metrics["bat_pressure_dots"] + prior_dot_rate * 30.0
) / (metrics["bat_pressure_balls_faced"] + 30.0)
metrics["bat_consistency_rate"] = metrics["bat_consistency_rate"].replace(0, np.nan)

league_bowl_legal = max(float(bowl_p["legal_ball"].sum()), 1.0)
prior_economy = float(bowl_p["bowler_runs_conceded"].sum() * 6.0 / league_bowl_legal)
prior_wicket_rate = float(bowl_p["bowler_credited_wicket"].sum() / league_bowl_legal * 100)
prior_bowl_dot_rate = float(bowl_p["bowler_dot"].sum() / league_bowl_legal)
prior_bowl_boundary_rate = float(bowl_p["bowler_boundary_conceded"].sum() / league_bowl_legal)

metrics["bowl_economy_raw"] = np.where(
    metrics["bowl_pressure_legal_balls"] > 0,
    metrics["bowl_pressure_runs_conceded"] * 6.0 / metrics["bowl_pressure_legal_balls"], np.nan
)
metrics["bowl_economy_shrunk"] = (
    metrics["bowl_pressure_runs_conceded"] + prior_economy * 18.0 / 6.0
) / ((metrics["bowl_pressure_legal_balls"] + 18.0) / 6.0)
metrics["bowl_wickets_per_100_shrunk"] = (
    metrics["bowl_pressure_wickets"] + prior_wicket_rate * 36.0 / 100.0
) / (metrics["bowl_pressure_legal_balls"] + 36.0) * 100.0
metrics["bowl_dot_rate_shrunk"] = (
    metrics["bowl_pressure_dots"] + prior_bowl_dot_rate * 36.0
) / (metrics["bowl_pressure_legal_balls"] + 36.0)
metrics["bowl_boundary_rate_shrunk"] = (
    metrics["bowl_pressure_boundaries"] + prior_bowl_boundary_rate * 36.0
) / (metrics["bowl_pressure_legal_balls"] + 36.0)
metrics["bowl_consistency_rate"] = metrics["bowl_consistency_rate"].replace(0, np.nan)

# Percentile helper: percentiles are calculated against the eligible league
# player pool, not just these 20 candidates.
def add_percentile(source, metric_col, out_col, higher_is_better=True, min_n_col=None, min_n=1):
    eligible = source.copy()
    if min_n_col is not None:
        eligible = eligible[eligible[min_n_col] >= min_n]
    vals = eligible[metric_col].replace([np.inf, -np.inf], np.nan).dropna()
    pct = vals.rank(pct=True, method="average", ascending=True) * 100
    if not higher_is_better:
        pct = 100 - pct
    source[out_col] = 50.0
    source.loc[pct.index, out_col] = pct
    return source

metrics = add_percentile(metrics, "bat_sr_shrunk", "bat_sr_pct", True, "bat_pressure_balls_faced", 20)
metrics = add_percentile(metrics, "bat_avg_shrunk", "bat_avg_pct", True, "bat_pressure_balls_faced", 20)
metrics = add_percentile(metrics, "bat_boundary_rate_shrunk", "bat_boundary_pct", True, "bat_pressure_balls_faced", 20)
metrics = add_percentile(metrics, "bat_dot_rate_shrunk", "bat_dot_pct", False, "bat_pressure_balls_faced", 20)
metrics = add_percentile(metrics, "bat_consistency_rate", "bat_consistency_pct", True, "bat_pressure_innings", 2)

metrics = add_percentile(metrics, "bowl_economy_shrunk", "bowl_economy_pct", False, "bowl_pressure_legal_balls", 12)
metrics = add_percentile(metrics, "bowl_wickets_per_100_shrunk", "bowl_wicket_pct", True, "bowl_pressure_legal_balls", 12)
metrics = add_percentile(metrics, "bowl_dot_rate_shrunk", "bowl_dot_pct", True, "bowl_pressure_legal_balls", 12)
metrics = add_percentile(metrics, "bowl_boundary_rate_shrunk", "bowl_boundary_pct", False, "bowl_pressure_legal_balls", 12)
metrics = add_percentile(metrics, "bowl_consistency_rate", "bowl_consistency_pct", True, "bowl_pressure_spells", 2)

# PLI is the leverage points generated per 100 successful comeback-pressure
# deliveries. A 0-100 percentile version is used only as a ranking component.
metrics["bat_pli_per_100_comeback_deliveries"] = np.where(
    metrics["bat_comeback_deliveries"] > 0,
    metrics["bat_pli_sum"] / metrics["bat_comeback_deliveries"], np.nan
)
metrics["bowl_pli_per_100_comeback_deliveries"] = np.where(
    metrics["bowl_comeback_deliveries"] > 0,
    metrics["bowl_pli_sum"] / metrics["bowl_comeback_deliveries"], np.nan
)
metrics = add_percentile(metrics, "bat_pli_per_100_comeback_deliveries", "bat_pli_pct", True,
                          "bat_comeback_deliveries", 5)
metrics = add_percentile(metrics, "bowl_pli_per_100_comeback_deliveries", "bowl_pli_pct", True,
                          "bowl_comeback_deliveries", 5)

# Reliability composites: role metrics first, then sample-size shrinkage toward
# a neutral 50. This stops tiny samples from dominating the player shortlist.
metrics["bat_score_raw"] = (
    0.30 * metrics["bat_sr_pct"] +
    0.25 * metrics["bat_avg_pct"] +
    0.15 * metrics["bat_boundary_pct"] +
    0.15 * metrics["bat_dot_pct"] +
    0.15 * metrics["bat_consistency_pct"]
)
metrics["bat_sample_confidence"] = 1.0 - np.exp(-metrics["bat_pressure_balls_faced"] / 60.0)
metrics["bat_score_adjusted"] = 50.0 + metrics["bat_sample_confidence"] * (
    metrics["bat_score_raw"] - 50.0
)
metrics["bat_pli_confidence"] = 1.0 - np.exp(-metrics["bat_comeback_deliveries"] / 20.0)
metrics["bat_pli_adjusted"] = 50.0 + metrics["bat_pli_confidence"] * (
    metrics["bat_pli_pct"] - 50.0
)
metrics["batting_reliability_score"] = (
    0.85 * metrics["bat_score_adjusted"] + 0.15 * metrics["bat_pli_adjusted"]
)

metrics["bowl_score_raw"] = (
    0.30 * metrics["bowl_economy_pct"] +
    0.25 * metrics["bowl_wicket_pct"] +
    0.20 * metrics["bowl_dot_pct"] +
    0.10 * metrics["bowl_boundary_pct"] +
    0.15 * metrics["bowl_consistency_pct"]
)
metrics["bowl_sample_confidence"] = 1.0 - np.exp(-metrics["bowl_pressure_legal_balls"] / 60.0)
metrics["bowl_score_adjusted"] = 50.0 + metrics["bowl_sample_confidence"] * (
    metrics["bowl_score_raw"] - 50.0
)
metrics["bowl_pli_confidence"] = 1.0 - np.exp(-metrics["bowl_comeback_deliveries"] / 20.0)
metrics["bowl_pli_adjusted"] = 50.0 + metrics["bowl_pli_confidence"] * (
    metrics["bowl_pli_pct"] - 50.0
)
metrics["bowling_reliability_score"] = (
    0.85 * metrics["bowl_score_adjusted"] + 0.15 * metrics["bowl_pli_adjusted"]
)

# Map IDs back to normalized candidate names.
candidate_id_map = {}
for candidate in CANDIDATES:
    ids = name_id.loc[name_id["name_norm"] == normalize_name(candidate), "player_id"].unique().tolist()
    candidate_id_map[candidate] = [int(x) for x in ids]

candidate_rows = []
for candidate in CANDIDATES:
    ids = candidate_id_map[candidate]
    if not ids:
        candidate_rows.append({
            "player": candidate, "player_ids": "", "role_from_dataset": "No dataset record",
            "total_batting_deliveries": 0, "total_bowling_deliveries": 0,
            "bat_pressure_balls_faced": 0, "bowl_pressure_legal_balls": 0,
            "bat_pressure_runs": 0, "bat_pressure_dismissals": 0,
            "bat_pressure_strike_rate": np.nan, "bat_pressure_average_shrunk": np.nan,
            "bat_pressure_boundary_rate": np.nan, "bat_pressure_dot_rate": np.nan,
            "batting_reliability_score": np.nan, "bat_comeback_matches": 0,
            "bat_comeback_balls": 0, "bat_pli_per_100_comeback_deliveries": np.nan,
            "bat_pli_points": 0, "bowl_pressure_runs_conceded": 0,
            "bowl_pressure_wickets": 0, "bowl_pressure_economy": np.nan,
            "bowl_pressure_wickets_per_100_balls": np.nan, "bowl_pressure_dot_rate": np.nan,
            "bowling_reliability_score": np.nan, "bowl_comeback_matches": 0,
            "bowl_comeback_legal_balls": 0,
            "bowl_pli_per_100_comeback_deliveries": np.nan, "bowl_pli_points": 0,
            "overall_reliability_score": np.nan, "sample_confidence": 0.0
        })
        continue
    rows = metrics[metrics["player_id"].isin(ids)]
    # Multiple IDs with the exact same normalized name are combined. In this
    # file candidate names resolve to a single ID in almost all cases.
    agg = rows.select_dtypes(include=[np.number]).sum(numeric_only=True)
    # Keep role scores as volume-weighted means if aliases map to multiple IDs.
    def weighted_score(score_col, weight_col):
        if score_col not in rows or weight_col not in rows:
            return np.nan
        w = rows[weight_col].clip(lower=0)
        if w.sum() <= 0:
            return np.nan
        return float(np.average(rows[score_col], weights=w))
    bat_score = weighted_score("batting_reliability_score", "bat_pressure_balls_faced")
    bowl_score = weighted_score("bowling_reliability_score", "bowl_pressure_legal_balls")
    bat_pli_val = weighted_score("bat_pli_per_100_comeback_deliveries", "bat_comeback_deliveries")
    bowl_pli_val = weighted_score("bowl_pli_per_100_comeback_deliveries", "bowl_comeback_deliveries")
    bat_total = int(rows["total_batting_deliveries"].sum())
    bowl_total = int(rows["total_bowling_deliveries"].sum())
    bat_pressure_balls = float(rows["bat_pressure_balls_faced"].sum())
    bowl_pressure_balls = float(rows["bowl_pressure_legal_balls"].sum())
    bat_conf = 1.0 - np.exp(-bat_pressure_balls / 60.0)
    bowl_conf = 1.0 - np.exp(-bowl_pressure_balls / 60.0)

    if bat_total >= 60 and bowl_total >= 60:
        role = "All-rounder"
        overall = np.nanmean([bat_score, bowl_score])
        confidence = np.mean([bat_conf, bowl_conf])
    elif bat_total >= 60:
        role = "Batter"
        overall, confidence = bat_score, bat_conf
    elif bowl_total >= 60:
        role = "Bowler"
        overall, confidence = bowl_score, bowl_conf
    elif bat_total + bowl_total > 0:
        role = "Limited sample / role uncertain"
        # Do not claim a robust role-specific score from a tiny volume.
        overall = np.nanmean([bat_score, bowl_score])
        confidence = max(bat_conf, bowl_conf) * 0.5
    else:
        role, overall, confidence = "No dataset record", np.nan, 0.0

    candidate_rows.append({
        "player": candidate,
        "player_ids": ";".join(map(str, ids)),
        "role_from_dataset": role,
        "total_batting_deliveries": bat_total,
        "total_bowling_deliveries": bowl_total,
        "bat_pressure_balls_faced": round(bat_pressure_balls, 1),
        "bowl_pressure_legal_balls": round(bowl_pressure_balls, 1),
        "bat_pressure_runs": float(rows["bat_pressure_runs"].sum()),
        "bat_pressure_dismissals": int(rows["bat_pressure_dismissals"].sum()),
        "bat_pressure_strike_rate": weighted_score("bat_sr_raw", "bat_pressure_balls_faced"),
        "bat_pressure_average_shrunk": weighted_score("bat_avg_shrunk", "bat_pressure_dismissals"),
        "bat_pressure_boundary_rate": weighted_score("bat_boundary_rate_shrunk", "bat_pressure_balls_faced"),
        "bat_pressure_dot_rate": weighted_score("bat_dot_rate_shrunk", "bat_pressure_balls_faced"),
        "batting_reliability_score": bat_score,
        "bat_comeback_matches": int(rows["bat_comeback_matches"].sum()),
        "bat_comeback_balls": float(rows["bat_comeback_balls"].sum()),
        "bat_comeback_deliveries": int(rows["bat_comeback_deliveries"].sum()),
        "bat_pli_per_100_comeback_deliveries": bat_pli_val,
        "bat_pli_points": float(rows["bat_pli_sum"].sum()),
        "bowl_pressure_runs_conceded": float(rows["bowl_pressure_runs_conceded"].sum()),
        "bowl_pressure_wickets": int(rows["bowl_pressure_wickets"].sum()),
        "bowl_pressure_economy": weighted_score("bowl_economy_raw", "bowl_pressure_legal_balls"),
        "bowl_pressure_wickets_per_100_balls": weighted_score("bowl_wickets_per_100_shrunk", "bowl_pressure_legal_balls"),
        "bowl_pressure_dot_rate": weighted_score("bowl_dot_rate_shrunk", "bowl_pressure_legal_balls"),
        "bowling_reliability_score": bowl_score,
        "bowl_comeback_matches": int(rows["bowl_comeback_matches"].sum()),
        "bowl_comeback_legal_balls": float(rows["bowl_comeback_legal_balls"].sum()),
        "bowl_comeback_deliveries": int(rows["bowl_comeback_deliveries"].sum()),
        "bowl_pli_per_100_comeback_deliveries": bowl_pli_val,
        "bowl_pli_points": float(rows["bowl_pli_sum"].sum()),
        "overall_reliability_score": overall,
        "sample_confidence": confidence
    })

rankings = pd.DataFrame(candidate_rows)

# Recalculate player-level overall score with robust confidence handling for
# candidates without pressure events / no historical rows.
def recommendation(row):
    if row["role_from_dataset"] == "No dataset record":
        return "Not assessable — no records in supplied CSV"
    if row["sample_confidence"] < 0.25:
        return "Watchlist — insufficient high-pressure sample"
    if row["overall_reliability_score"] >= 75 and row["sample_confidence"] >= 0.60:
        return "Priority shortlist — strong historical pressure evidence"
    if row["overall_reliability_score"] >= 65 and row["sample_confidence"] >= 0.40:
        return "Consider — positive pressure profile"
    if row["overall_reliability_score"] >= 50:
        return "Conditional — average/mixed evidence; assess role and price"
    return "Low priority — below the pressure-performance benchmark"

rankings["auction_recommendation_performance_only"] = rankings.apply(recommendation, axis=1)
rankings["data_quality_note"] = rankings.apply(
    lambda r: (
        "No player deliveries found in CSV."
        if r["role_from_dataset"] == "No dataset record" else
        "Small sample: avoid strong conclusions."
        if r["sample_confidence"] < 0.25 else
        "Historical delivery sample available; still verify 2027 availability and squad fit."
    ), axis=1
)

# All 20 names are retained. No-record candidates appear last and remain
# explicitly unscored rather than receiving an invented performance score.
rankings["_has_score"] = rankings["overall_reliability_score"].notna()
rankings = rankings.sort_values(
    ["_has_score", "overall_reliability_score", "sample_confidence"],
    ascending=[False, False, False]
).reset_index(drop=True)
rankings.insert(0, "rank", np.arange(1, len(rankings) + 1))
rankings = rankings.drop(columns=["_has_score"])



## Cell 8: AUCTION RECOMMENDATIONS


In [9]:
# The recommendation is deliberately labelled PERFORMANCE-ONLY. The CSV
# does not contain 2027 availability, auction eligibility, expected price or
# current RR squad needs. Those public-data inputs can be added in phase 2.
auction_recommendations = rankings[[
    "rank", "player", "role_from_dataset", "overall_reliability_score",
    "batting_reliability_score", "bowling_reliability_score",
    "bat_pli_per_100_comeback_deliveries", "bowl_pli_per_100_comeback_deliveries",
    "sample_confidence", "auction_recommendation_performance_only",
    "data_quality_note"
]].copy()
auction_recommendations["ipl_2027_availability"] = "To verify from public sources"
auction_recommendations["rr_squad_fit"] = "Not assessed — squad/retention data not yet added"
auction_recommendations["auction_value"] = "Not assessed — price/budget data not yet added"

# Add concise summary of pressure metrics for interpretation.
print("\nCandidate rankings (performance-only):")
cols_show = ["rank", "player", "role_from_dataset", "overall_reliability_score",
             "sample_confidence", "auction_recommendation_performance_only"]
print(rankings[cols_show].to_string(index=False))

print("\nKey caveat: ranking is based only on this CSV's historical delivery records.")
print("It does not assess 2027 availability, auction eligibility, current RR squad fit or price.")





Candidate rankings (performance-only):
 rank               player               role_from_dataset  overall_reliability_score  sample_confidence                     auction_recommendation_performance_only
    1 Mahendra Singh Dhoni                          Batter                  84.607430           0.993591    Priority shortlist — strong historical pressure evidence
    2          Virat Kohli                          Batter                  80.625362           0.977253    Priority shortlist — strong historical pressure evidence
    3       Dinesh Karthik                          Batter                  79.954884           0.981684    Priority shortlist — strong historical pressure evidence
    4       Faf du Plessis                          Batter                  78.490209           0.959238    Priority shortlist — strong historical pressure evidence
    5         Rohit Sharma                          Batter                  71.639097           0.990596                        Conside

## Cell 9: EXPORT FILES


In [10]:
OUTPUT_DIR = "/content/rr_auction_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

rankings.to_csv(os.path.join(OUTPUT_DIR, "rr_player_rankings.csv"), index=False)
auction_recommendations.to_csv(
    os.path.join(OUTPUT_DIR, "rr_auction_recommendations.csv"), index=False
)
validation.to_csv(os.path.join(OUTPUT_DIR, "rr_win_probability_validation.csv"), index=False)
match_results.to_csv(os.path.join(OUTPUT_DIR, "rr_inferred_match_results.csv"), index=False)

# Export the pressure-delivery rows for explainability / later dashboard work.
pressure_cols = [
    "match_id", "innings", "innings_ball_num", "striker_name", "bowler_name",
    "batting_team_id", "bowling_team_id", "winner_team_id", "total_runs",
    "is_wicket", "wp_batting_pre", "wp_batting_post", "wp_bowling_pre",
    "batting_pressure", "bowling_pressure", "batting_comeback_pressure",
    "bowling_comeback_pressure", "bat_pli_event", "bowl_pli_event"
]
pressure_events = wp.loc[
    wp["batting_pressure"] | wp["bowling_pressure"],
    [c for c in pressure_cols if c in wp.columns]
].copy()
pressure_events.to_csv(os.path.join(OUTPUT_DIR, "rr_pressure_delivery_events.csv"), index=False)

zip_path = "/content/rr_auction_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for filename in os.listdir(OUTPUT_DIR):
        zf.write(os.path.join(OUTPUT_DIR, filename), arcname=filename)

print("\nCreated outputs:")
for name in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", os.path.join(OUTPUT_DIR, name))
print("ZIP:", zip_path)

import os
os.startfile(os.path.dirname(os.path.abspath(zip_path)))


# ============================================================
# ============================================================



Created outputs:
 - /content/rr_auction_outputs\rr_auction_recommendations.csv
 - /content/rr_auction_outputs\rr_inferred_match_results.csv
 - /content/rr_auction_outputs\rr_player_rankings.csv
 - /content/rr_auction_outputs\rr_pressure_delivery_events.csv
 - /content/rr_auction_outputs\rr_win_probability_validation.csv
ZIP: /content/rr_auction_outputs.zip
